# 04 - LDA Topic Modeling

**SMATIC 7.0 - Pipeline Tahap 4**

Notebook ini membentuk topik dari teks hasil notebook 03 menggunakan
`sklearn.decomposition.LatentDirichletAllocation`.

Prinsip metodologis:
- jumlah topik dipilih dengan coherence NPMI, stabilitas antarseed, topic diversity, dan held-out perplexity;
- corpus pembentukan model diseimbangkan per wilayah agar volume Jakarta tidak menenggelamkan Banten;
- seluruh tweet tetap mendapat distribusi probabilitas topik;
- prevalensi dilaporkan dengan soft assignment dan `source_weight`;
- nama substantif topik tetap ditentukan manual dari kata kunci dan tweet representatif.

Input wajib: `data_for_lda.csv` dari notebook 03.
Output utama: `lda_results.csv`, `lda_topic_summary.csv`, dan `organized_csv/06_lda/`.

## 1. Setup dan Validasi Input

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import warnings

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

warnings.filterwarnings('default')

INPUT_PATH = Path('data_for_lda.csv')
OUTPUT_DIR = Path('organized_csv/06_lda')
AUDIT_DIR = OUTPUT_DIR / 'audits'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = {
    'id', 'text', 'text_clean', 'text_stemmed', '_region', '_search_keyword',
    'account_class', 'source_weight', 'sentiment_label', 'createdAt',
}

if not INPUT_PATH.exists():
    raise FileNotFoundError('data_for_lda.csv belum ada. Jalankan notebook 03 terlebih dahulu.')

df_raw = pd.read_csv(INPUT_PATH)
missing_columns = sorted(REQUIRED_COLUMNS - set(df_raw.columns))
if missing_columns:
    raise ValueError(f'Kolom wajib belum tersedia: {missing_columns}')

df_raw['source_weight'] = pd.to_numeric(df_raw['source_weight'], errors='coerce')
if df_raw['source_weight'].isna().any() or (df_raw['source_weight'] <= 0).any():
    raise ValueError('source_weight harus numerik dan lebih besar dari 0.')
if df_raw['id'].duplicated().any():
    raise ValueError(f'Ditemukan {df_raw["id"].duplicated().sum()} ID duplikat. Perbaiki notebook 03.')

print(f'Total input: {len(df_raw):,} tweet')
print(f'Distribusi wilayah: {df_raw["_region"].value_counts().to_dict()}')
print(f'Missing text_stemmed: {df_raw["text_stemmed"].isna().sum():,}')

## 2. Persiapan Corpus dan Vocabulary

In [ ]:
# Region dihapus dari vocabulary karena sudah tersedia sebagai metadata dan dapat
# menciptakan topik semu yang hanya membedakan kata "Jakarta" vs "Banten".
DOMAIN_STOPWORDS = {
    'yang', 'dan', 'dengan', 'untuk', 'dari', 'pada', 'dalam', 'di', 'ke', 'ini', 'itu',
    'ada', 'jadi', 'juga', 'sudah', 'akan', 'bisa', 'saya', 'kamu', 'kami',
    'kita', 'mereka', 'dia', 'sangat', 'banget', 'sekali', 'lebih', 'paling',
    'masih', 'tapi', 'karena', 'kalau', 'ketika', 'setelah', 'sebelum',
    'sama', 'semua', 'setiap', 'banyak', 'sedikit', 'lagi', 'aja', 'nih', 'ya',
    'sih', 'dong', 'deh', 'kok', 'rt', 'via', 'amp', 'link', 'https',
    'sch', 'fess', 'menfess', 'sender', 'admin', 'min', 'base',
    'sekolah', 'pendidikan', 'pendidik', 'didik', 'siswa', 'murid', 'pelajar',
    # Anchor pencarian dihapus agar LDA tidak sekadar menemukan ulang keyword query.
    'ppdb', 'spmb', 'kjp', 'kjmu', 'ukt', 'bos', 'beasiswa', 'kuliah', 'akses', 'zonasi',
    # Token percakapan sangat umum yang tidak membedakan isu.
    'mau', 'buat', 'orang', 'tahun', 'satu', 'pak', 'name', 'trivia', 'nya',
    'udah', 'kalo', 'beri', 'perintah',
    'jakarta', 'banten',
}

def basic_tokens(text):
    return re.findall(r'(?u)\b\w\w+\b', str(text).lower())

df = df_raw.copy()
df['_tokens_before_vectorizer'] = df['text_stemmed'].fillna('').apply(
    lambda text: [token for token in basic_tokens(text) if token not in DOMAIN_STOPWORDS]
)
short_mask = df['_tokens_before_vectorizer'].apply(len) < 3
short_document_audit = df[short_mask].copy()
df = df[~short_mask].copy().reset_index(drop=True)

def near_duplicate_signature(text):
    tokens = re.findall(r'(?u)\b\w\w+\b', str(text).lower())
    if len(tokens) < 20:
        return ''
    return ' '.join(tokens[:20])

df['_near_duplicate_signature'] = df['text_clean'].fillna(df['text']).apply(
    near_duplicate_signature
)
near_duplicate_mask = (
    df['_near_duplicate_signature'].ne('')
    & df.duplicated('_near_duplicate_signature', keep='first')
)
near_duplicate_audit = df[near_duplicate_mask].copy()
df = df[~near_duplicate_mask].copy().reset_index(drop=True)

print(f'Dokumen terlalu pendek (<3 token): {len(short_document_audit):,}')
print(f'Near-duplicate dikeluarkan: {len(near_duplicate_audit):,}')
print(f'Dokumen kandidat LDA: {len(df):,}')
print(f'Median token: {df["_tokens_before_vectorizer"].apply(len).median():.0f}')

# Pembentukan model diseimbangkan per wilayah. Semua dokumen tetap diinferensikan nanti.
MODEL_SENTIMENTS = {'Negatif', 'Positif'}
model_candidate_df = df[
    df['sentiment_label'].isin(MODEL_SENTIMENTS)
    & df['sentiment_review_status'].eq('accepted')
].copy()
if model_candidate_df['_region'].nunique() < 2:
    raise ValueError('Corpus aspirasi non-netral tidak mencakup kedua wilayah.')

MAX_FIT_DOCS_PER_KEYWORD = 200
query_capped_df = pd.concat(
    [
        group.sample(min(len(group), MAX_FIT_DOCS_PER_KEYWORD), random_state=42)
        for _, group in model_candidate_df.groupby(['_region', '_search_keyword'], sort=True)
    ],
    ignore_index=True,
)
REGION_COUNTS = query_capped_df['_region'].value_counts()
FIT_DOCS_PER_REGION = int(REGION_COUNTS.min())

def sample_region_for_model(group, target_size, region):
    if len(group) <= target_size:
        return group.copy()
    try:
        sampled, _ = train_test_split(
            group,
            train_size=target_size,
            random_state=42,
            stratify=group['_search_keyword'],
        )
        return sampled
    except ValueError as error:
        warnings.warn(f'Stratified sampling gagal untuk {region}: {error}. Memakai random sampling.')
        return group.sample(target_size, random_state=42)

model_df = pd.concat(
    [
        sample_region_for_model(group, FIT_DOCS_PER_REGION, region)
        for region, group in query_capped_df.groupby('_region', sort=True)
    ],
    ignore_index=True,
).sample(frac=1, random_state=42).reset_index(drop=True)

# Pooling meningkatkan co-occurrence untuk LDA pada teks pendek. Model dibentuk dari
# pseudo-document, tetapi inferensi akhir tetap dilakukan pada setiap tweet.
POOL_SIZE = 10
CORPUS_CONFIG_VERSION = 'query-aligned-pooled-v4'
def build_pseudo_documents(data, pool_size):
    rows = []
    for (region, keyword), group in data.groupby(['_region', '_search_keyword'], sort=True):
        group = group.sample(frac=1, random_state=42).reset_index(drop=True)
        for start in range(0, len(group), pool_size):
            chunk = group.iloc[start:start + pool_size]
            if len(chunk) < 2:
                continue
            rows.append({
                '_region': region,
                '_search_keyword': keyword,
                'text_stemmed': ' '.join(chunk['text_stemmed'].fillna('')),
                'n_tweets': len(chunk),
            })
    return pd.DataFrame(rows)

pooled_model_df = build_pseudo_documents(model_df, POOL_SIZE)

train_df, validation_df = train_test_split(
    pooled_model_df,
    test_size=0.20,
    random_state=42,
    stratify=pooled_model_df['_region'],
)

MIN_DF = max(2, int(np.ceil(len(train_df) * 0.01)))
MAX_DF = 0.80
vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=sorted(DOMAIN_STOPWORDS),
    token_pattern=r'(?u)\b\w\w+\b',
    ngram_range=(1, 2),
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=6000,
)

X_train = vectorizer.fit_transform(train_df['text_stemmed'].fillna(''))
X_validation = vectorizer.transform(validation_df['text_stemmed'].fillna(''))
X_model = vectorizer.transform(pooled_model_df['text_stemmed'].fillna(''))
X_all = vectorizer.transform(df['text_stemmed'].fillna(''))
feature_names = vectorizer.get_feature_names_out()

zero_vector_mask = np.asarray(X_all.sum(axis=1)).ravel() == 0
zero_vector_audit = df[zero_vector_mask].copy()
if zero_vector_mask.any():
    df = df[~zero_vector_mask].reset_index(drop=True)
    X_all = X_all[~zero_vector_mask]

print(f'Cap per keyword untuk model: {MAX_FIT_DOCS_PER_KEYWORD}')
print(f'Kandidat model (accepted non-netral): {len(model_candidate_df):,}')
print(f'Tweet pembentuk model: {len(model_df):,} ({FIT_DOCS_PER_REGION:,}/wilayah)')
print(f'Pseudo-document (pool={POOL_SIZE}): {len(pooled_model_df):,}')
print(f'Train/validation: {X_train.shape[0]:,}/{X_validation.shape[0]:,}')
print(f'Vocabulary unigram+bigram: {len(feature_names):,}')
print(f'Zero-vector dikeluarkan: {len(zero_vector_audit):,}')

## 3. Pemilihan Jumlah Topik

K dievaluasi pada data validasi. Coherence NPMI menjadi kriteria utama,
dengan guardrail topic diversity dan stabilitas antarseed. Held-out perplexity
dilaporkan sebagai diagnostik; nilai terendah tidak otomatis berarti topik paling mudah ditafsirkan.

In [ ]:
TOP_WORDS_EVAL = 10
EXPECTED_CORPUS_CONFIG_VERSION = 'query-aligned-pooled-v4'
if globals().get('CORPUS_CONFIG_VERSION') != EXPECTED_CORPUS_CONFIG_VERSION:
    raise RuntimeError('Konfigurasi corpus stale. Restart kernel dan jalankan notebook dari atas.')
K_VALUES = range(2, 8)
EVALUATION_SEEDS = (42, 137)
EVALUATION_MAX_ITER = 15
MIN_TOPIC_DIVERSITY = 0.70
MIN_TOPIC_STABILITY = 0.45

def top_word_indices(model, top_n=10):
    return np.argsort(model.components_, axis=1)[:, -top_n:][:, ::-1]

def topic_diversity(model, top_n=10):
    indices = top_word_indices(model, top_n)
    return len(np.unique(indices)) / indices.size

def npmi_coherence(model, document_term_matrix, top_n=10):
    binary = document_term_matrix.astype(bool).astype(np.int8).tocsr()
    n_docs = binary.shape[0]
    doc_freq = np.asarray(binary.sum(axis=0)).ravel()
    topic_scores = []

    for topic_words in top_word_indices(model, top_n):
        pair_scores = []
        for i in range(len(topic_words)):
            for j in range(i + 1, len(topic_words)):
                wi, wj = topic_words[i], topic_words[j]
                cooc = binary[:, wi].multiply(binary[:, wj]).sum()
                if cooc == 0 or doc_freq[wi] == 0 or doc_freq[wj] == 0:
                    pair_scores.append(-1.0)
                    continue
                p_i = doc_freq[wi] / n_docs
                p_j = doc_freq[wj] / n_docs
                p_ij = cooc / n_docs
                pair_scores.append(np.log(p_ij / (p_i * p_j)) / -np.log(p_ij))
        topic_scores.append(np.mean(pair_scores))
    return float(np.mean(topic_scores))

def topic_stability(model_a, model_b, top_n=10):
    sets_a = [set(words) for words in top_word_indices(model_a, top_n)]
    sets_b = [set(words) for words in top_word_indices(model_b, top_n)]
    similarity = np.array([
        [len(a & b) / max(len(a | b), 1) for b in sets_b]
        for a in sets_a
    ])
    rows, cols = linear_sum_assignment(-similarity)
    return float(similarity[rows, cols].mean())

def train_lda(k, seed, X, max_iter):
    model = LatentDirichletAllocation(
        n_components=k,
        learning_method='batch',
        max_iter=max_iter,
        random_state=seed,
        doc_topic_prior=1.0 / k,
        topic_word_prior=1.0 / k,
        evaluate_every=-1,
        n_jobs=-1,
    )
    return model.fit(X)

evaluation_rows = []
candidate_models = {}

for k in K_VALUES:
    seed_models = [
        train_lda(k, seed, X_train, EVALUATION_MAX_ITER)
        for seed in EVALUATION_SEEDS
    ]
    stability_models = [
        train_lda(k, seed, X_model, EVALUATION_MAX_ITER)
        for seed in EVALUATION_SEEDS
    ]
    candidate_models[k] = seed_models
    coherence_values = [
        npmi_coherence(model, X_validation, TOP_WORDS_EVAL)
        for model in seed_models
    ]
    perplexity_values = [model.perplexity(X_validation) for model in seed_models]
    diversity_values = [topic_diversity(model, TOP_WORDS_EVAL) for model in stability_models]
    stability = topic_stability(stability_models[0], stability_models[1], TOP_WORDS_EVAL)

    evaluation_rows.append({
        'k': k,
        'coherence_npmi': np.mean(coherence_values),
        'coherence_std': np.std(coherence_values),
        'heldout_perplexity': np.mean(perplexity_values),
        'topic_diversity': np.mean(diversity_values),
        'topic_stability': stability,
        'stability_scope': 'full_pooled_corpus',
    })
    print(
        f'K={k}: coherence={np.mean(coherence_values):.3f}, '
        f'stability={stability:.3f}, diversity={np.mean(diversity_values):.3f}, '
        f'perplexity={np.mean(perplexity_values):.1f}'
    )

evaluation_df = pd.DataFrame(evaluation_rows)
eligible = evaluation_df[
    (evaluation_df['topic_diversity'] >= MIN_TOPIC_DIVERSITY) &
    (evaluation_df['topic_stability'] >= MIN_TOPIC_STABILITY)
]
if eligible.empty:
    evaluation_df.to_csv(OUTPUT_DIR / 'topic_count_evaluation_failed.csv', index=False)
    raise RuntimeError(
        'Tidak ada K yang lolos guardrail diversity dan stability. '
        'Jangan interpretasikan topik; bersihkan corpus atau revisi sampling terlebih dahulu.'
    )

BEST_K = int(
    eligible.sort_values(
        ['coherence_npmi', 'topic_stability', 'heldout_perplexity'],
        ascending=[False, False, True],
    ).iloc[0]['k']
)
evaluation_df['selected'] = evaluation_df['k'].eq(BEST_K)
print(f'\nK terpilih: {BEST_K}')
display(evaluation_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(evaluation_df['k'], evaluation_df['coherence_npmi'], marker='o')
axes[0].set_title('Coherence NPMI (lebih tinggi lebih baik)')
axes[1].plot(evaluation_df['k'], evaluation_df['topic_stability'], marker='o', color='#287271')
axes[1].axhline(MIN_TOPIC_STABILITY, linestyle='--', color='gray')
axes[1].set_title('Stabilitas Antarseed')
axes[2].plot(evaluation_df['k'], evaluation_df['heldout_perplexity'], marker='o', color='#b56576')
axes[2].set_title('Held-out Perplexity (lebih rendah lebih baik)')
for axis in axes:
    axis.axvline(BEST_K, linestyle='--', color='#c1121f', alpha=0.8)
    axis.set_xlabel('Jumlah topik (K)')
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'topic_count_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. Model Final dan Kata Kunci Topik

In [ ]:
FINAL_K = BEST_K  # Boleh diubah setelah inspeksi substantif.
FINAL_K_OVERRIDE_REASON = ''
if FINAL_K not in set(evaluation_df['k']):
    raise ValueError(f'FINAL_K={FINAL_K} belum pernah dievaluasi pada K_VALUES.')
if FINAL_K != BEST_K and not FINAL_K_OVERRIDE_REASON.strip():
    raise ValueError('Isi FINAL_K_OVERRIDE_REASON ketika FINAL_K berbeda dari BEST_K.')
FINAL_SEEDS = (42, 137, 2026)
FINAL_MAX_ITER = 30

final_candidates = [
    train_lda(FINAL_K, seed, X_model, FINAL_MAX_ITER)
    for seed in FINAL_SEEDS
]
final_quality = pd.DataFrame({
    'seed': FINAL_SEEDS,
    'coherence_npmi': [npmi_coherence(model, X_model, TOP_WORDS_EVAL) for model in final_candidates],
    'topic_diversity': [topic_diversity(model, TOP_WORDS_EVAL) for model in final_candidates],
    'perplexity_fit': [model.perplexity(X_model) for model in final_candidates],
})
best_seed_row = final_quality.sort_values(
    ['coherence_npmi', 'topic_diversity', 'perplexity_fit'],
    ascending=[False, False, True],
).iloc[0]
FINAL_SEED = int(best_seed_row['seed'])
lda_model = final_candidates[list(FINAL_SEEDS).index(FINAL_SEED)]
data_signature = hashlib.sha1(
    pd.util.hash_pandas_object(df[['id', 'text_stemmed']], index=False).values.tobytes()
).hexdigest()[:12]
MODEL_SIGNATURE = (
    f'{data_signature}_k{FINAL_K}_seed{FINAL_SEED}_v{len(feature_names)}_pool{POOL_SIZE}'
)

TOP_WORDS_REPORT = 12
topic_summary_rows = []
topic_keywords = {}
for topic_id, indices in enumerate(top_word_indices(lda_model, TOP_WORDS_REPORT)):
    weights = lda_model.components_[topic_id, indices]
    weights = weights / weights.sum()
    topic_keywords[topic_id] = feature_names[indices].tolist()
    for rank, (index, weight) in enumerate(zip(indices, weights), start=1):
        topic_summary_rows.append({
            'topic_id': topic_id,
            'rank': rank,
            'keyword': feature_names[index],
            'within_top_words_weight': float(weight),
        })

topic_summary_df = pd.DataFrame(topic_summary_rows)
print(f'Final K={FINAL_K}, seed={FINAL_SEED}')
display(final_quality.round(4))
for topic_id, words in topic_keywords.items():
    print(f'Topik {topic_id}: {", ".join(words)}')

## 5. Distribusi Topik per Tweet dan Audit Ketidakpastian

In [ ]:
topic_probabilities = lda_model.transform(X_all)
probability_columns = [f'topic_{topic_id}_probability' for topic_id in range(FINAL_K)]
probability_df = pd.DataFrame(topic_probabilities, columns=probability_columns, index=df.index)
df = pd.concat([df, probability_df], axis=1)

sorted_probabilities = np.sort(topic_probabilities, axis=1)
df['topic_id'] = topic_probabilities.argmax(axis=1)
df['topic_probability'] = topic_probabilities.max(axis=1)
df['topic_margin'] = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
df['topic_entropy'] = (
    -np.sum(topic_probabilities * np.log(np.clip(topic_probabilities, 1e-12, 1)), axis=1)
    / np.log(FINAL_K)
)

TOPIC_PROBABILITY_THRESHOLD = 0.35
TOPIC_MARGIN_THRESHOLD = 0.10
df['topic_review_status'] = np.where(
    (df['topic_probability'] < TOPIC_PROBABILITY_THRESHOLD) |
    (df['topic_margin'] < TOPIC_MARGIN_THRESHOLD),
    'needs_manual_review',
    'accepted',
)

draft_names = {
    topic_id: f'Topik {topic_id}: ' + ' / '.join(words[:3])
    for topic_id, words in topic_keywords.items()
}
df['topic_name'] = df['topic_id'].map(draft_names)

doc_topic_long = pd.DataFrame({
    'id': np.repeat(df['id'].to_numpy(), FINAL_K),
    'topic_id': np.tile(np.arange(FINAL_K), len(df)),
    'topic_probability': topic_probabilities.ravel(),
    'source_weight': np.repeat(df['source_weight'].to_numpy(), FINAL_K),
})
doc_topic_long['weighted_topic_mass'] = (
    doc_topic_long['topic_probability'] * doc_topic_long['source_weight']
)

print(df['topic_review_status'].value_counts().to_string())
print(f'Rata-rata dominant probability: {df["topic_probability"].mean():.3f}')
print(f'Rata-rata normalized entropy: {df["topic_entropy"].mean():.3f}')

## 6. Tweet Representatif dan Penamaan Manual

In [ ]:
representative_pool = df[
    df['sentiment_label'].isin(MODEL_SENTIMENTS)
    & df['sentiment_review_status'].eq('accepted')
].copy()
representative_rows = []
for topic_id in range(FINAL_K):
    probability_column = f'topic_{topic_id}_probability'
    representatives = representative_pool.nlargest(8, probability_column)
    for rank, (_, row) in enumerate(representatives.iterrows(), start=1):
        representative_rows.append({
            'topic_id': topic_id,
            'rank': rank,
            'topic_probability': row[probability_column],
            'id': row['id'],
            'region': row['_region'],
            'keyword_query': row['_search_keyword'],
            'text': row['text'],
        })
representative_df = pd.DataFrame(representative_rows)

CONCEPTUAL_TOPIC_GUIDE = pd.DataFrame([
    {'code': 'A', 'candidate_label': 'Sistem Zonasi dan PPDB'},
    {'code': 'B', 'candidate_label': 'Biaya Pendidikan (UKT/BOS)'},
    {'code': 'C', 'candidate_label': 'Akses Pendidikan Daerah Terpencil'},
    {'code': 'D', 'candidate_label': 'Program Beasiswa dan Bantuan (KJP/KJMU)'},
    {'code': 'E', 'candidate_label': 'Ketimpangan Sekolah Negeri vs Swasta'},
])
print('Panduan konseptual (bukan mapping otomatis ke topic ID):')
display(CONCEPTUAL_TOPIC_GUIDE)

TOPIC_LABEL_PATH = Path('topic_labels_manual.csv')
manual_labels = pd.DataFrame([
        {
            'topic_id': topic_id,
            'top_keywords': ', '.join(topic_keywords[topic_id]),
            'representative_text_1': representative_df.query('topic_id == @topic_id').iloc[0]['text'],
            'representative_text_2': representative_df.query('topic_id == @topic_id').iloc[1]['text'],
            'model_signature': MODEL_SIGNATURE,
            'manual_topic_name': '',
        }
    for topic_id in range(FINAL_K)
])
if TOPIC_LABEL_PATH.exists():
    previous_labels = pd.read_csv(TOPIC_LABEL_PATH)
    if not {'topic_id', 'manual_topic_name'}.issubset(previous_labels.columns):
        raise ValueError('topic_labels_manual.csv wajib memiliki topic_id dan manual_topic_name.')
    same_model = (
        'model_signature' in previous_labels.columns
        and set(previous_labels['model_signature'].dropna()) == {MODEL_SIGNATURE}
    )
    if same_model:
        previous_name_map = (
            previous_labels.assign(
                manual_topic_name=previous_labels['manual_topic_name'].fillna('').str.strip()
            ).set_index('topic_id')['manual_topic_name'].to_dict()
        )
        manual_labels['manual_topic_name'] = manual_labels['topic_id'].map(previous_name_map).fillna('')
    else:
        warnings.warn('Model/corpus berubah; nama topik lama tidak dipakai karena topic ID dapat bergeser.')
manual_labels.to_csv(TOPIC_LABEL_PATH, index=False)
print(f'Template penamaan diperbarui: {TOPIC_LABEL_PATH}')

valid_manual_names = manual_labels.set_index('topic_id')['manual_topic_name'].to_dict()
final_names = {
    topic_id: valid_manual_names.get(topic_id) or draft_names[topic_id]
    for topic_id in range(FINAL_K)
}
df['topic_name'] = df['topic_id'].map(final_names)
topic_summary_df['topic_name'] = topic_summary_df['topic_id'].map(final_names)
representative_df['topic_name'] = representative_df['topic_id'].map(final_names)

for topic_id in range(FINAL_K):
    print(f'\n{final_names[topic_id]}')
    print('Kata kunci:', ', '.join(topic_keywords[topic_id]))
    for text in representative_df.query('topic_id == @topic_id').head(3)['text']:
        print('-', str(text)[:180])

## 7. Agregasi Berbobot per Wilayah, Keyword, dan Sentimen

In [ ]:
def weighted_soft_distribution(data, group_columns):
    rows = []
    grouped = data.groupby(group_columns, dropna=False)
    for group_key, group in grouped:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        weights = group['source_weight'].to_numpy()
        probabilities = group[probability_columns].to_numpy()
        weighted_mass = (probabilities * weights[:, None]).sum(axis=0)
        proportions = weighted_mass / weighted_mass.sum()
        base = dict(zip(group_columns, group_key))
        for topic_id, (mass, proportion) in enumerate(zip(weighted_mass, proportions)):
            rows.append({
                **base,
                'topic_id': topic_id,
                'topic_name': final_names[topic_id],
                'weighted_topic_mass': mass,
                'weighted_proportion': proportion,
                'weighted_percentage': proportion * 100,
                'n_documents': len(group),
            })
    return pd.DataFrame(rows)

topic_by_region = weighted_soft_distribution(df, ['_region'])
topic_by_keyword = weighted_soft_distribution(df, ['_region', '_search_keyword'])
topic_by_sentiment = weighted_soft_distribution(df, ['sentiment_label'])
topic_by_region_sentiment = weighted_soft_distribution(df, ['_region', 'sentiment_label'])
topic_by_account_class = weighted_soft_distribution(df, ['account_class'])

# Profil sentimen di dalam masing-masing topik: denominator = total massa topik.
topic_sentiment_mass = topic_by_sentiment.pivot(
    index=['topic_id', 'topic_name'],
    columns='sentiment_label',
    values='weighted_topic_mass',
).fillna(0)
topic_sentiment_profile = (
    topic_sentiment_mass.div(topic_sentiment_mass.sum(axis=1), axis=0) * 100
).reset_index()

accepted_sentiment_df = (
    df[df['sentiment_review_status'].eq('accepted')]
    if 'sentiment_review_status' in df.columns
    else df
)
if accepted_sentiment_df.empty:
    warnings.warn('Tidak ada prediksi sentimen accepted; sensitivity check memakai seluruh data.')
    accepted_sentiment_df = df
topic_by_sentiment_accepted = weighted_soft_distribution(
    accepted_sentiment_df, ['sentiment_label']
)
accepted_sentiment_mass = topic_by_sentiment_accepted.pivot(
    index=['topic_id', 'topic_name'], columns='sentiment_label', values='weighted_topic_mass'
).fillna(0)
topic_sentiment_profile_accepted = (
    accepted_sentiment_mass.div(accepted_sentiment_mass.sum(axis=1), axis=0) * 100
).reset_index()

print('=== DISTRIBUSI TOPIK BERBOBOT PER WILAYAH (%) ===')
region_pivot = topic_by_region.pivot(
    index='_region', columns='topic_name', values='weighted_percentage'
)
display(region_pivot.round(2))

print('=== TOPIK DENGAN SENTIMEN NEGATIF TERTINGGI ===')
if 'Negatif' in topic_sentiment_profile.columns:
    display(topic_sentiment_profile.sort_values('Negatif', ascending=False)[
        ['topic_name', 'Negatif']
    ].round(2))
print('=== SENSITIVITY CHECK: HANYA SENTIMEN ACCEPTED ===')
if 'Negatif' in topic_sentiment_profile_accepted.columns:
    display(topic_sentiment_profile_accepted.sort_values('Negatif', ascending=False)[
        ['topic_name', 'Negatif']
    ].round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(max(9, FINAL_K * 1.6), 4))
heatmap_data = region_pivot.to_numpy()
image = ax.imshow(heatmap_data, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(region_pivot.shape[1]))
ax.set_xticklabels(region_pivot.columns, rotation=30, ha='right')
ax.set_yticks(range(region_pivot.shape[0]))
ax.set_yticklabels(region_pivot.index)
for row in range(region_pivot.shape[0]):
    for column in range(region_pivot.shape[1]):
        ax.text(column, row, f'{heatmap_data[row, column]:.1f}%', ha='center', va='center')
fig.colorbar(image, ax=ax, label='Weighted soft-topic percentage')
ax.set_title('Distribusi Topik Berbobot: Jakarta vs Banten')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'topic_distribution_by_region.png', dpi=180, bbox_inches='tight')
plt.show()

## 8. Simpan Model, Audit, dan Tabel Analisis

In [ ]:
result_columns = [
    'id', 'text', 'text_stemmed', '_region', '_search_keyword', 'account_class',
    'source_weight', 'sentiment_label', 'sentiment_score', 'sentiment_review_status',
    'topic_id', 'topic_name', 'topic_probability', 'topic_margin', 'topic_entropy',
    'topic_review_status', 'createdAt',
] + probability_columns
result_columns = [column for column in result_columns if column in df.columns]

df[result_columns].to_csv('lda_results.csv', index=False)
df[result_columns].to_csv(OUTPUT_DIR / 'lda_results.csv', index=False)
topic_summary_df.to_csv('lda_topic_summary.csv', index=False)
topic_summary_df.to_csv(OUTPUT_DIR / 'lda_topic_summary.csv', index=False)
evaluation_df.to_csv(OUTPUT_DIR / 'topic_count_evaluation.csv', index=False)
final_quality.to_csv(OUTPUT_DIR / 'final_seed_evaluation.csv', index=False)
doc_topic_long.to_csv(OUTPUT_DIR / 'document_topic_probabilities_long.csv', index=False)
representative_df.to_csv(OUTPUT_DIR / 'representative_tweets.csv', index=False)
manual_labels.to_csv(OUTPUT_DIR / 'topic_labels_manual.csv', index=False)
CONCEPTUAL_TOPIC_GUIDE.to_csv(OUTPUT_DIR / 'conceptual_topic_guide.csv', index=False)
topic_by_region.to_csv(OUTPUT_DIR / 'weighted_topics_by_region.csv', index=False)
topic_by_keyword.to_csv(OUTPUT_DIR / 'weighted_topics_by_keyword.csv', index=False)
topic_by_sentiment.to_csv(OUTPUT_DIR / 'weighted_topics_by_sentiment.csv', index=False)
topic_by_region_sentiment.to_csv(
    OUTPUT_DIR / 'weighted_topics_by_region_sentiment.csv', index=False
)
topic_by_account_class.to_csv(OUTPUT_DIR / 'weighted_topics_by_account_class.csv', index=False)
topic_sentiment_profile.to_csv(OUTPUT_DIR / 'sentiment_profile_within_topic.csv', index=False)
topic_sentiment_profile_accepted.to_csv(
    OUTPUT_DIR / 'sentiment_profile_within_topic_accepted_only.csv', index=False
)
model_df.groupby(['_region', '_search_keyword']).size().reset_index(name='n_fit').to_csv(
    OUTPUT_DIR / 'model_fit_composition.csv', index=False
)

short_document_audit.to_csv(AUDIT_DIR / 'excluded_short_documents.csv', index=False)
near_duplicate_audit.to_csv(AUDIT_DIR / 'excluded_near_duplicates.csv', index=False)
zero_vector_audit.to_csv(AUDIT_DIR / 'excluded_zero_vector_documents.csv', index=False)
df[df['topic_review_status'].eq('needs_manual_review')].to_csv(
    AUDIT_DIR / 'ambiguous_topic_assignments.csv', index=False
)

joblib.dump(lda_model, OUTPUT_DIR / 'lda_model.joblib')
joblib.dump(vectorizer, OUTPUT_DIR / 'count_vectorizer.joblib')

run_metadata = {
    'algorithm': 'sklearn.decomposition.LatentDirichletAllocation',
    'sklearn_version': sklearn.__version__,
    'input_path': str(INPUT_PATH),
    'n_input': int(len(df_raw)),
    'n_analyzed': int(len(df)),
    'n_near_duplicates_excluded': int(len(near_duplicate_audit)),
    'model_sentiments': sorted(MODEL_SENTIMENTS),
    'n_model_fit_tweets': int(len(model_df)),
    'n_pseudo_documents': int(len(pooled_model_df)),
    'pool_size': POOL_SIZE,
    'fit_docs_per_region': FIT_DOCS_PER_REGION,
    'max_fit_docs_per_keyword': MAX_FIT_DOCS_PER_KEYWORD,
    'vocabulary_size': int(len(feature_names)),
    'min_df': MIN_DF,
    'max_df': MAX_DF,
    'ngram_range': [1, 2],
    'selected_k': FINAL_K,
    'final_k_override_reason': FINAL_K_OVERRIDE_REASON or None,
    'selected_seed': FINAL_SEED,
    'model_signature': MODEL_SIGNATURE,
    'final_max_iter': FINAL_MAX_ITER,
    'topic_probability_threshold': TOPIC_PROBABILITY_THRESHOLD,
    'topic_margin_threshold': TOPIC_MARGIN_THRESHOLD,
    'prevalence_method': 'soft topic probability multiplied by source_weight',
}
(OUTPUT_DIR / 'run_metadata.json').write_text(
    json.dumps(run_metadata, indent=2), encoding='utf-8'
)
(OUTPUT_DIR / 'README.md').write_text(
    '# LDA Outputs\n\n'
    'Model dibentuk dari corpus seimbang per wilayah, lalu seluruh tweet diinferensikan.\n'
    'Gunakan weighted_topics_by_region.csv untuk prevalensi wilayah dan '
    'sentiment_profile_within_topic.csv untuk persentase sentimen di dalam topik.\n'
    'Assignment ambigu tersedia di audits/ambiguous_topic_assignments.csv.\n',
    encoding='utf-8',
)

print(f'Saved: lda_results.csv ({len(df):,} tweet)')
print(f'Saved organized outputs: {OUTPUT_DIR}')
print(f'Final K={FINAL_K}, seed={FINAL_SEED}')

## Catatan Metodologis untuk Esai

> Analisis topik dilakukan dengan Latent Dirichlet Allocation (LDA) menggunakan
> implementasi scikit-learn dan representasi bag-of-words unigram-bigram. Karena tweet
> merupakan teks pendek, tweet near-duplicate dikeluarkan dan kelompok kecil tweet non-netral
> dengan prediksi sentimen accepted dari wilayah dan query yang sama digabung
> menjadi pseudo-document hanya untuk pembentukan model; inferensi tetap dilakukan per tweet. Corpus
> pembentukan model diseimbangkan pada jumlah dokumen yang sama per wilayah agar
> ketimpangan volume Jakarta dan Banten tidak menentukan struktur topik. Model yang
> terbentuk kemudian digunakan untuk mengestimasi distribusi topik seluruh tweet.
> Jumlah topik dipilih menggunakan coherence NPMI sebagai kriteria utama, disertai
> evaluasi stabilitas antarseed, topic diversity, dan held-out perplexity. Prevalensi
> topik dihitung dengan soft assignment, yaitu menjumlahkan probabilitas setiap topik
> setelah dikalikan bobot sumber, bukan hanya menghitung label topik dominan. Nama
> substantif topik ditentukan manual melalui kata kunci dan tweet representatif.

Batasan interpretasi:
- topik menunjukkan pola pada corpus hasil pencarian keyword, bukan prevalensi populasi;
- label wilayah adalah wilayah query, bukan bukti domisili pengguna;
- LDA menggunakan bag-of-words sehingga tidak memahami ironi atau urutan kalimat;
- assignment berprobabilitas rendah atau margin kecil wajib diaudit manual.
- jumlah pseudo-document per wilayah masih terbatas; coherence dan stability dipakai sebagai diagnostik eksploratif, bukan bukti validitas populasi.